In [1]:
# Cell 1: Install Required Packages (Run Once)
!pip install -q chromadb sentence-transformers transformers

# Cell 2: Imports
import chromadb
from sentence_transformers import SentenceTransformer
from pathlib import Path
import pandas as pd
import json
import os

print("✅ Imports loaded!")

# Cell 3: Configuration
# Set project root (notebook is in notebooks/ folder, so go up one level)
PROJECT_ROOT = Path.cwd().parent
VECTOR_STORE_PATH = PROJECT_ROOT / "vector_store" / "chroma_db"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
TOP_K = 5

print(f"Vector store path: {VECTOR_STORE_PATH}")
print(f"Exists? {VECTOR_STORE_PATH.exists()}")

# Cell 4: ComplaintRAG Class
class ComplaintRAG:
    def __init__(self, vector_store_path=None, embedding_model=None):
        if vector_store_path is None:
            vector_store_path = VECTOR_STORE_PATH
        if embedding_model is None:
            embedding_model = EMBEDDING_MODEL
        
        # Load embedding model
        self.model = SentenceTransformer(embedding_model)
        
        # Load vector store
        self.client = chromadb.PersistentClient(path=str(vector_store_path))
        self.collection = self.client.get_collection("complaints")
        
        print(f"✅ Loaded vector store with {self.collection.count()} chunks")
    
    def retrieve(self, query, top_k=None):
        """Retrieve top-k relevant chunks for a query"""
        if top_k is None:
            top_k = TOP_K
        
        # Generate query embedding
        query_embedding = self.model.encode([query])[0]
        
        # Search in vector store
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )
        
        # Extract documents and metadata
        documents = results['documents'][0] if results['documents'] else []
        metadatas = results['metadatas'][0] if results['metadatas'] else []
        distances = results['distances'][0] if results['distances'] else []
        
        retrieved = []
        for doc, meta, dist in zip(documents, metadatas, distances):
            retrieved.append({
                'text': doc,
                'complaint_id': meta.get('complaint_id', 'Unknown'),
                'product': meta.get('product', 'Unknown'),
                'score': 1 - dist  # convert distance to similarity score
            })
        
        return retrieved
    
    def generate_answer(self, query, retrieved_chunks, llm_function):
        """Generate answer using retrieved context and LLM"""
        # Build context from retrieved chunks
        context_parts = []
        for i, chunk in enumerate(retrieved_chunks):
            context_parts.append(f"[Source {i+1}] Product: {chunk['product']}\n{chunk['text']}")
        
        context = "\n\n---\n\n".join(context_parts)
        
        # Create prompt
        prompt = f"""You are a financial analyst assistant for CreditTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.

Context:
{context}

Question: {query}

Answer:"""
        
        # Generate response using the provided LLM function
        response = llm_function(prompt)
        return response
    
    def query(self, query, llm_function, top_k=None):
        """Full RAG pipeline: retrieve + generate"""
        # 1. Retrieve
        retrieved = self.retrieve(query, top_k)
        
        # 2. Generate
        answer = self.generate_answer(query, retrieved, llm_function)
        
        return {
            'question': query,
            'answer': answer,
            'sources': retrieved
        }

print("✅ ComplaintRAG class defined!")

# Cell 5: LLM Functions
def use_huggingface_local(prompt, model_name="HuggingFaceH4/zephyr-7b-beta"):
    """Use a local Hugging Face model"""
    from transformers import pipeline
    generator = pipeline("text-generation", model=model_name, device="cpu")
    result = generator(prompt, max_new_tokens=200, do_sample=False)
    return result[0]['generated_text']

def use_huggingface_api(prompt, model="mistralai/Mistral-7B-Instruct-v0.3"):
    """Use Hugging Face Inference API (free tier)"""
    import requests
    
    API_TOKEN = os.getenv("HF_TOKEN")
    if not API_TOKEN:
        print("⚠️ HF_TOKEN not set. Using mock LLM.")
        return mock_llm(prompt)
    
    API_URL = f"https://api-inference.huggingface.co/models/{model}"
    headers = {"Authorization": f"Bearer {API_TOKEN}"}
    
    payload = {
        "inputs": prompt,
        "parameters": {"max_new_tokens": 200, "temperature": 0.3}
    }
    
    try:
        response = requests.post(API_URL, headers=headers, json=payload, timeout=30)
        if response.status_code == 200:
            return response.json()[0]['generated_text']
        else:
            print(f"⚠️ API Error: {response.status_code}")
            return mock_llm(prompt)
    except Exception as e:
        print(f"⚠️ API Exception: {e}")
        return mock_llm(prompt)

def mock_llm(prompt):
    """Placeholder for testing"""
    return f"[Mock Answer] The system retrieved relevant complaint excerpts. Please set up a real LLM for actual answers."

print("✅ LLM functions defined!")

# Cell 6: Initialize RAG System
print("Initializing RAG system...")
rag = ComplaintRAG()
print("✅ RAG system ready!")

# Cell 7: Define Test Questions
test_questions = [
    "Why are people unhappy with Credit Cards?",
    "What are the most common complaints about Money Transfers?",
    "Are there any complaints about unauthorized transactions?",
    "What billing issues do customers report?",
    "Do customers complain about customer service?",
    "Are there any complaints about fees?",
    "What fraud-related complaints do customers have?",
]

print(f"✅ {len(test_questions)} test questions defined:")
for i, q in enumerate(test_questions, 1):
    print(f"   {i}. {q}")

# Cell 8: Choose LLM Function
# Choose which LLM function to use:

# Option 1: Mock LLM (fastest, no API key needed)
llm_function = mock_llm
print("Using: Mock LLM (for testing)")

# Option 2: Hugging Face API (uncomment this and set HF_TOKEN)
# llm_function = use_huggingface_api
# print("Using: Hugging Face API")

# Option 3: Local LLM (uncomment this, requires ~7GB RAM)
# llm_function = use_huggingface_local
# print("Using: Local LLM")

# Cell 9: Evaluation Function
def evaluate_rag(rag_system, test_questions, llm_function):
    """Run evaluation on test questions"""
    results = []
    
    for i, question in enumerate(test_questions, 1):
        print(f"\n{'='*60}")
        print(f"Test Question {i}/{len(test_questions)}")
        print(f"Q: {question}")
        print('-'*60)
        
        result = rag_system.query(question, llm_function)
        
        print(f"A: {result['answer']}")
        print(f"\nSources: {len(result['sources'])} chunks retrieved")
        if result['sources']:
            print(f"Top source score: {result['sources'][0]['score']:.3f}")
            print(f"Top source product: {result['sources'][0]['product']}")
        
        results.append(result)
    
    return results

print("✅ Evaluation function defined!")

# Cell 10: Run Evaluation
print("\n" + "="*60)
print("Running RAG Evaluation")
print("="*60)

results = evaluate_rag(rag, test_questions, llm_function)

# Cell 11: Summary Table
print("\n" + "="*60)
print("Evaluation Summary Table")
print("="*60)

print("\n| # | Question | Sources | Top Score | Product |")
print("|---|----------|---------|-----------|---------|")
for i, r in enumerate(results, 1):
    top_score = f"{r['sources'][0]['score']:.3f}" if r['sources'] else "N/A"
    top_product = r['sources'][0]['product'] if r['sources'] else "N/A"
    question_short = r['question'][:30] + "..." if len(r['question']) > 30 else r['question']
    print(f"| {i} | {question_short} | {len(r['sources'])} | {top_score} | {top_product} |")

# Cell 12: Detailed Results
print("\n" + "="*60)
print("Detailed Results")
print("="*60)

for i, r in enumerate(results, 1):
    print(f"\n{i}. Question: {r['question']}")
    print(f"\n   Answer: {r['answer']}")
    print(f"\n   Sources ({len(r['sources'])} total):")
    for j, source in enumerate(r['sources'][:3], 1):  # Show top 3 sources
        print(f"      Source {j}: Product={source['product']}, Score={source['score']:.3f}")
        print(f"      Text: {source['text'][:150]}...")
        print()

# Cell 13: Test a Single Custom Question
# Run this cell to test any question you want
custom_question = "What do customers say about credit card fees?"
print(f"Testing: {custom_question}")
print("-" * 60)

result = rag.query(custom_question, llm_function)

print(f"\nAnswer: {result['answer']}")
print(f"\nSources: {len(result['sources'])} chunks retrieved")
for i, source in enumerate(result['sources'][:3], 1):
    print(f"\nSource {i}: Product={source['product']}, Score={source['score']:.3f}")
    print(f"Text: {source['text'][:200]}...")

# Cell 14: Save Results to JSON (Optional)
output_data = {
    "test_questions": test_questions,
    "results": [
        {
            "question": r["question"],
            "answer": r["answer"],
            "sources": r["sources"]
        }
        for r in results
    ]
}

with open("../reports/evaluation_results.json", "w") as f:
    json.dump(output_data, f, indent=2)

print("✅ Results saved to reports/evaluation_results.json")


✅ Imports loaded!
Vector store path: C:\Users\user\OneDrive\Desktop\Project\KAIM\credittrust-complaint-rag\credittrust-complaint-rag\vector_store\chroma_db
Exists? True
✅ ComplaintRAG class defined!
✅ LLM functions defined!
Initializing RAG system...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Loaded vector store with 5852 chunks
✅ RAG system ready!
✅ 7 test questions defined:
   1. Why are people unhappy with Credit Cards?
   2. What are the most common complaints about Money Transfers?
   3. Are there any complaints about unauthorized transactions?
   4. What billing issues do customers report?
   5. Do customers complain about customer service?
   6. Are there any complaints about fees?
   7. What fraud-related complaints do customers have?
Using: Mock LLM (for testing)
✅ Evaluation function defined!

Running RAG Evaluation

Test Question 1/7
Q: Why are people unhappy with Credit Cards?
------------------------------------------------------------
A: [Mock Answer] The system retrieved relevant complaint excerpts. Please set up a real LLM for actual answers.

Sources: 5 chunks retrieved
Top source score: 0.576
Top source product: Credit card

Test Question 2/7
Q: What are the most common complaints about Money Transfers?
---------------------------------------------------